# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do NOT subscript; access as object properties
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"FAIR citation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect and print out the available Record Sets and their fields, referencing them by their `@id` only. This overview helps us select which data to extract and process in later steps.

In [ ]:
# List record sets by @id
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"  - RecordSet @id: {rs['@id']}  |  Name: {rs.get('name','(no name)')}")
    record_set_ids.append(rs['@id'])

# Show fields for each record set
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"Fields (@id):")
    for f in fields:
        if isinstance(f, dict):
            print(f"  - {f.get('@id','[Unknown]')} (Name: {f.get('name','')})")
        else:
            print(f"  - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Use the record set and field `@id`s from the overview above.

We extract all rows for each record set by its `@id`, and store them as DataFrames for analysis.

In [ ]:
# Extract data for each record set
all_dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        all_dataframes[rs_id] = df
        print(f"RecordSet @id: {rs_id}  - Loaded {len(df)} records.")
        print(f"Fields (DataFrame columns): {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not extract records for {rs_id}: {e}")
        continue

# Example: show the first DataFrame's content
if record_set_ids:
    first_id = record_set_ids[0]
    display_cols = all_dataframes[first_id].columns.tolist()
    print(f"\nShowing first 5 records for record set {first_id}:")
    display(all_dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

For demonstration, we select a numeric field from one record set, filter on it, normalize, and group if a suitable categorical field exists. **All fields are referenced by their `@id` (as column name)**.

In [ ]:
# Example EDA on the first record set loaded
import numpy as np

# User: Select the RecordSet and the numeric/categorical field ids manually based on previous output
# For this dataset, suppose the main data is in the first record set, and example numeric field '@id's are: 'accession', 'coverage', 'unique_peptides', 'protein_mw', etc.

record_set_id = record_set_ids[0] if record_set_ids else None
df = all_dataframes[record_set_id]

# If the actual columns are, e.g.: ['accession', 'description', 'coverage', 'unique_peptides', 'protein_mw', 'pl', ...]
# We'll pick a likely numeric field. Let's print the columns for reference.
print("Fields for selected DataFrame:")
print(df.columns.tolist())

# Choose a numeric field, such as 'coverage' if available. Adjust below to match actual field '@id'.
numeric_field = None
candidate_numeric_fields = ['coverage', 'unique_peptides', 'protein_mw', 'pl', 'psm', 'score']
for f in candidate_numeric_fields:
    if f in df.columns:
        numeric_field = f
        break

if numeric_field is None:
    raise Exception("No suitable numeric field found in this record set.")

threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0

# Ensure data is numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field such as 'accession' or 'description' or any other
group_field = None
candidate_group_fields = ['description', 'accession', 'gene_name', 'cluster', 'category']
for f in candidate_group_fields:
    if f in df.columns and df[f].nunique() < 25:
        group_field = f
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical field with <25 unique values for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If there is a second numeric field, show scatter plot
other_numeric = [col for col in df.columns if col != numeric_field and np.issubdtype(df[col].dtype, np.number)]
if other_numeric:
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=df[numeric_field], y=df[other_numeric[0]])
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric[0])
    plt.title(f'Scatter: {numeric_field} vs {other_numeric[0]}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Dataset provided detailed records on proteins detected from extracellular vesicles of human mast cells.
- We loaded metadata, examined available record sets and their fields (by `@id`), extracted data as DataFrames, and explored numeric measurements.
- Filtering, normalization, grouping, and visualizations demonstrated how to process and inspect Croissant datasets with `mlcroissant`.

For more, consult [mlcroissant documentation](https://mlcommons.github.io/croissant/spec/) and explore the schema details at the dataset's Croissant URL.